# 01 — Explore the Corner Maze

Interactive manual control of the corner-maze environment. Use this to build intuition for the maze layout, the action space, and what one trial cycle looks like.

## Setup

- **Local (VS Code):** `pip install -e .` from the repo root, then optionally `scripts/setup_data.sh` to copy the eye-image parquet from the legacy repo (the maze panel works without it; eye images will show a placeholder if the parquet is missing).
- **Colab:** the install cell below installs the package directly from GitHub.

## What you'll see

- **Maze** — top-down 13×13 view with the agent, barriers, cues, and wells.
- **Left / Right Eye** — first-person grayscale views from the agent's pose (only if the eye-image parquet is present).
- **Info panel** — current position, direction, step count, and cumulative reward.

## Controls

| Key | Action |
|---|---|
| ↑ | Forward |
| ← / → | Turn left / right |
| ↓ | Pause |
| Space | Enter well (only legal at corner positions) |
| r | Reset |

*Click the maze to focus before pressing keys.*


In [1]:
# Setup: install the package on Colab; just import it locally.
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # --upgrade --force-reinstall so re-running this cell after a `git push`
    # actually picks up the new code (pip otherwise skips already-installed
    # versions, and the kernel-cached module would persist anyway — restart
    # the runtime if you've already imported the package).
    !pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/ryangrg/corner-maze-rl.git
    !pip install -q ipywidgets ipyevents

import numpy as np
from corner_maze_rl.env.corner_maze_env import CornerMazeEnv
from corner_maze_rl.env.constants import CORNER_POSES, EYE_IMG_SIZE
from minigrid.core.actions import Actions

print("Imports OK.")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Imports OK.


## Pick a session type

The cell below uses `PI+VC f2 acquisition` by default. Other session types you can swap in:

**Exposure (open maze, no barriers):**
- `exposure` — Exposure A: open maze, all wells rewarded
- `exposure_b` — Exposure B: progressive barrier lowering

**PI+VC combined (fixed second turn, f2):**
- `PI+VC f2 acquisition` · `PI+VC f2 novel route` · `PI+VC f2 no cue` · `PI+VC f2 rotate` · `PI+VC f2 reversal`

**PI+VC combined (fixed first turn, f1):**
- `PI+VC f1 acquisition` · `PI+VC f1 novel route` · `PI+VC f1 no cue` · `PI+VC f1 rotate` · `PI+VC f1 reversal`

**PI-only (path integration, no visual cue):**
- `PI acquisition` · `PI novel route no cue` · `PI novel route cue` · `PI reversal no cue` · `PI reversal cue`

**VC-only (visual cue, rotating across arms):**
- `VC acquisition` · `VC novel route fixed` · `VC novel route rotate` · `VC reversal fixed` · `VC reversal rotate`


In [ ]:
#@title Interactive manual control
import io
import urllib.request
from pathlib import Path

import ipywidgets as widgets
from ipyevents import Event
from IPython.display import display
from PIL import Image as PILImage

# --- Configure the env here ---
SESSION_TYPE = "PI+VC f2 acquisition"
ORIENTATION = "N/NE"
GOAL = "NE"
SEED = 42

# --- Stage the eye-image parquet so the env can find it ---
# The 2 MB embeddings parquet isn't bundled with the pip wheel. We pull it
# from the legacy repo (public, raw URL) into ./data/dataframes/ on first
# run so the env's _load_embeddings() picks it up via its CWD-search path.
EYE_PARQUET = Path("data/dataframes/dual-indep-20260319-222411-embeddings-allposes.parquet")
EYE_URL = ("https://github.com/ryangrg/corner-maze-rl-legacy/raw/main/"
           "data/dataframes/dual-indep-20260319-222411-embeddings-allposes.parquet")
if not EYE_PARQUET.is_file():
    EYE_PARQUET.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading eye-image embeddings (~2 MB) → {EYE_PARQUET}")
    urllib.request.urlretrieve(EYE_URL, EYE_PARQUET)
    print(f"  done ({EYE_PARQUET.stat().st_size // 1024} KB)")

env = CornerMazeEnv(
    render_mode="rgb_array",
    max_steps=10000,
    session_type=SESSION_TYPE,
    agent_cue_goal_orientation=ORIENTATION,
    start_goal_location=GOAL,
)
obs, info = env.reset(seed=SEED)

HAS_EYES = False
try:
    env._load_embeddings()
    HAS_EYES = True
except Exception as e:
    print(f"(eye images unavailable: {type(e).__name__}; maze panel still works)")

# --- Theme ---
BG_COLOR = "#333333"
TEXT_COLOR = "#ffffff"
MUTED_COLOR = "#aaaaaa"
DIR_NAMES = {0: "East", 1: "South", 2: "West", 3: "North"}
IMG_SIZE = 299  # 13 * 23 = 299, avoids grid-line aliasing

style_widget = widgets.HTML(f"""<style>
.maze-ui, .maze-ui * {{
    background-color: {BG_COLOR} !important;
    color: {TEXT_COLOR} !important;
    border-color: {BG_COLOR} !important;
}}
.maze-ui {{
    padding: 15px !important;
    border-radius: 8px !important;
    margin: 0 !important;
}}
.maze-ui .widget-image {{
    background-color: transparent !important;
}}
</style>""")

# --- State ---
cumulative_reward = 0.0
is_done = False
zero_eye = np.zeros((EYE_IMG_SIZE, EYE_IMG_SIZE), dtype=np.float64)


def frame_to_png_bytes(frame):
    img = PILImage.fromarray(frame).resize((IMG_SIZE, IMG_SIZE), PILImage.NEAREST)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


# Eye images live in [0,1] but most highlights cap around 0.3, so a raw
# (arr*255) cast looks too dark. We apply a fixed gain so bright details
# saturate cleanly while dim poses stay dim — i.e. comparable across
# frames. Per-image min/max stretching looks better on average frames
# but blows up uniformly-dark poses (max ~0.01) into noisy white. Fixed
# gain avoids that.
EYE_GAIN = 3.0


def eye_to_png_bytes(eye_arr):
    """Render a [0,1]-ish float eye image as PNG with fixed-gain scaling."""
    a = np.asarray(eye_arr, dtype=np.float64)
    gray = (np.clip(a * EYE_GAIN, 0.0, 1.0) * 255).astype(np.uint8)
    img = PILImage.fromarray(gray).resize((IMG_SIZE, IMG_SIZE), PILImage.NEAREST)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


# --- Widgets ---
maze_widget = widgets.Image(format="png", width=IMG_SIZE, height=IMG_SIZE)
left_eye_widget = widgets.Image(format="png", width=IMG_SIZE, height=IMG_SIZE)
right_eye_widget = widgets.Image(format="png", width=IMG_SIZE, height=IMG_SIZE)
session_label = widgets.HTML(value="")
image_label = widgets.HTML(value="")
info_label = widgets.HTML(value="")


def update_display():
    global cumulative_reward, is_done

    frame = env.get_allocentric_frame(tile_size=23)
    maze_widget.value = frame_to_png_bytes(frame)

    if HAS_EYES:
        pose_label = env._get_pose_label()
        left_eye = env._pose_to_left_eye.get(pose_label, zero_eye)
        right_eye = env._pose_to_right_eye.get(pose_label, zero_eye)
    else:
        pose_label = "(eye images not loaded)"
        left_eye = zero_eye
        right_eye = zero_eye
    left_eye_widget.value = eye_to_png_bytes(left_eye)
    right_eye_widget.value = eye_to_png_bytes(right_eye)

    session_label.value = (
        f'<div style="font-size:13px; text-align:center; margin-top:4px;">'
        f'<b>{env.session_type}</b></div>'
    )
    image_label.value = (
        f'<div style="font-size:13px; text-align:center; margin-top:4px;">'
        f'<code>{pose_label}</code></div>'
    )

    x, y = env.agent_pos
    d = env.agent_dir
    info_label.value = (
        f'<div style="font-size:14px; line-height:1.8;">'
        f'<b>Position:</b> ({x}, {y})<br>'
        f'<b>Direction:</b> {d} ({DIR_NAMES.get(d, "?")})<br>'
        f'<b>Step:</b> {env.step_count} / {env.max_steps}<br>'
        f'<b>Reward:</b> {cumulative_reward:.4f}<br>'
        f'<b>At corner:</b> {(x, y, d) in CORNER_POSES}<br>'
        f'<b>Done:</b> {is_done}'
        f'</div>'
    )


def on_action(action):
    global cumulative_reward, is_done
    if is_done:
        return
    obs, reward, terminated, truncated, info = env.step(action)
    cumulative_reward += reward
    is_done = terminated or truncated
    update_display()


def on_reset():
    global cumulative_reward, is_done
    cumulative_reward = 0.0
    is_done = False
    env.reset()
    update_display()


KEY_ACTION_MAP = {
    "ArrowUp": Actions.forward,
    "ArrowLeft": Actions.left,
    "ArrowRight": Actions.right,
    "ArrowDown": 4,  # ACTION_PAUSE
    " ": 3,  # ACTION_ENTER_WELL (space)
}


def on_key(event):
    key = event.get("key", "")
    if key == "r":
        on_reset()
        return
    action = KEY_ACTION_MAP.get(key)
    if action is not None:
        on_action(action)


# --- Layout ---
panel_style = widgets.Layout(flex="1 1 0%", align_items="center")
maze_panel = widgets.VBox(
    [widgets.HTML("<b>Maze</b>"), maze_widget, session_label],
    layout=panel_style,
)
left_eye_panel = widgets.VBox(
    [widgets.HTML("<b>Left Eye</b>"), left_eye_widget], layout=panel_style
)
right_eye_panel = widgets.VBox(
    [widgets.HTML("<b>Right Eye</b>"), right_eye_widget], layout=panel_style
)
eye_panel = widgets.VBox(
    [
        widgets.HBox(
            [left_eye_panel, right_eye_panel],
            layout=widgets.Layout(width="100%"),
        ),
        image_label,
    ],
    layout=widgets.Layout(flex="2 1 0%", align_items="center"),
)
images_row = widgets.HBox(
    [maze_panel, eye_panel],
    layout=widgets.Layout(width="100%", margin="0 0 6px 0"),
)

if "_kb_event" in dir():
    try:
        _kb_event.close()
    except Exception:
        pass
_kb_event = Event(source=images_row, watched_events=["keydown"], prevent_default_action=True)
_kb_event.on_dom_event(on_key)

info_panel = widgets.VBox(
    [
        widgets.HTML(
            f'<div style="font-size:12px; color:{MUTED_COLOR}; margin-bottom:8px;">'
            f"<b>Keyboard:</b> Click maze to focus, then use arrow keys.<br>"
            f"Up=Forward, Left/Right=Turn, Down=Pause, Space=Enter Well, r=Reset"
            f"</div>"
        ),
        info_label,
    ],
    layout=widgets.Layout(padding="10px"),
)

ui = widgets.VBox([style_widget, images_row, info_panel])
ui.add_class("maze-ui")
update_display()
display(ui)


## Try this

1. Run a complete trial: navigate from a start arm through one of the four corners and into the correct well (Space at the corner pose).
2. Watch how `Reward` changes — small negative per step (`-0.0005` forward, `-0.001` turn) and a `+1.061` spike on a correct well.
3. Try a wrong well and see that the trial doesn't end — the env keeps running until you visit the correct one.
4. Swap `SESSION_TYPE` for `"exposure"` and notice that all four wells reward (cycle-alternation).

## What's next

- **02_explore_yoked_data** — load the rodent behavioral dataset and plot trajectories.
- **03_compute_returns** — replay yoked sessions through the env to attach per-step reward and per-trial-with-ITI-start RTG.
- **04_train_dt** — train the Decision Transformer on `actions_with_returns.parquet`.
